In [52]:
import os
from wsgiref import headers

import pandas as pd
import functools as ft
import holidays
data_path = os.path.join(os.getcwd(), "..", "data")

load_data = pd.read_csv(os.path.join(data_path, "big_blue_load_forecasts.csv"))
res_data = pd.read_csv(os.path.join(data_path, "big_blue_res_forecasts.csv"))
ppts_data = pd.read_csv(os.path.join(data_path, "big_blue_ppts.csv"))
border_data = pd.read_csv(os.path.join(data_path, "big_blue_borders.csv"))
curt_data = pd.read_csv(os.path.join(data_path, "big_blue_curtailements.csv"))

In [53]:
def merge_manip(left:pd.DataFrame, right:pd.DataFrame, on:list, how:str):
    if left.empty:
        return right
    if right.empty:
        return left
    return pd.merge(left, right, on=on, how=how)

In [54]:
data_df = ft.reduce(lambda left, right: merge_manip(left, right, on="datetime_from", how="left"),
                          [load_data, res_data, ppts_data, border_data, curt_data.fillna(0)])
data_df["datetime_from"] = [pd.to_datetime(x).tz_convert("cet") for x in data_df["datetime_from"]]
data_df["holiday"] = [x.date() for x in data_df["datetime_from"]]
data_df["holiday"] = [1 if x in holidays.Greece(years=[2025, 2026]).keys() else 0 for x in data_df["holiday"]]

In [56]:
data_df.to_csv(os.path.join(data_path, "data_sample.csv"), index=False, header=True)